In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Remove display limits for comprehensive auditing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)  # Prevents text truncation in cells

In [2]:
raw_data_dir = "../data/raw/2025"
csv_files = glob.glob(os.path.join(raw_data_dir, "*.csv"))

schemas = {}
for file in csv_files:
    month_name = os.path.basename(file)
    # Read only nrows=1 for analysis
    df_head = pd.read_csv(file, sep=';', encoding='latin-1', nrows=1, low_memory=False)
    schemas[month_name] = set(df_head.columns.str.strip())

print(f"Analyzed Files: {len(schemas)}")

Analyzed Files: 12


In [3]:
# Columns that are in ALL files (reliable cols)
universal_cols = set.intersection(*schemas.values())

# What are all possible columns that exists?
all_possible_cols = set.union(*schemas.values())

# Wich cells appears and dissapears? (dangerous cols)
drifting_cols = all_possible_cols - universal_cols

print(f"Total of unique fields in a the year: {len(all_possible_cols)}")
print(f"Universal fields (safe): {len(universal_cols)}")
print(f"Variable fields (unstable): {len(drifting_cols)}")

print("\n--- UNSTABLE COLUMNS ---")
for col in sorted(list(drifting_cols)):
    print(f"- {col}")

Total of unique fields in a the year: 78
Universal fields (safe): 78
Variable fields (unstable): 0

--- UNSTABLE COLUMNS ---


In [4]:
# Shows file name, column quantity and headers 
print("--- AUDITORÍA DE CABECERAS POR MES ---")

for file_name, cols in sorted(schemas.items()):
    print(f"\n[{file_name}] - {len(cols)} columnas:")
    
    columnas_ordenadas = sorted(list(cols))
    print(", ".join(columnas_ordenadas))

--- AUDITORÍA DE CABECERAS POR MES ---

[2025-1.csv] - 78 columnas:
ActividadComprador, ActividadProveedor, CantidadEvaluacion, Cargos, Categoria, CiudadUnidadCompra, Codigo, CodigoAbreviadoTipoOC, CodigoLicitacion, CodigoOrganismoPublico, CodigoProveedor, CodigoSucursal, CodigoTipo, CodigoUnidadCompra, Codigo_ConvenioMarco, ComunaProveedor, Descripcion/Obervaciones, DescripcionTipoOC, Descuentos, EsCompraAgil, EsTratoDirecto, EspecificacionComprador, EspecificacionProveedor, Estado, EstadoProveedor, FechaAceptacion, FechaCancelacion, FechaCreacion, FechaEnvio, FechaSolicitudCancelacion, Financiamiento, Forma de Pago, FormaPago, ID, IDItem, Impuestos, Link, MontoTotalOC, MontoTotalOC_PesosChilenos, Nombre, NombreProveedor, NombreroductoGenerico, OrganismoPublico, Pais, PaisProveedor, PaisUnidadCompra, PorcentajeIva, ProcedenciaOC, PromedioCalificacion, RegionProveedor, RegionUnidadCompra, RubroN1, RubroN2, RubroN3, RutSucursal, RutUnidadCompra, Sucursal, Tipo, TipoDespacho, TipoImpuest

In [5]:
# Taking the first file as sample to analyze datatypes
# This time we read 10,000 rows to infer datatypes
sample_file = csv_files[0]
df_sample = pd.read_csv(sample_file, sep=';', encoding='latin-1', nrows=10000, low_memory=False)

# Check if there are columns that have to be numbers but show as strings (ex. amounts with commas)
data_types = pd.DataFrame({
    'DataType': df_sample.dtypes,
    'Null_values': df_sample.isnull().sum(),
    'Null_percentage': (df_sample.isnull().sum() / len(df_sample)) * 100
}).sort_values('Null_percentage', ascending=False)

display(data_types)

,DataType,Null_values,Null_percentage
FechaCancelacion,float64,10000,100.00
Codigo_ConvenioMarco,object,9623,96.23
Financiamiento,object,4542,45.42
CodigoLicitacion,object,3692,36.92
PaisProveedor,object,3229,32.29
EspecificacionProveedor,object,2577,25.77
totalImpuestos,float64,2381,23.81
ActividadComprador,object,1952,19.52
Pais,object,498,4.98
PaisUnidadCompra,object,498,4.98


In [6]:
"""
Data Structure Inspection
=========================
Purpose: Samples raw records to inspect actual data formats, anomalies, and string structures.
"""

# Extract a random sample of 10 records for unbiased inspection
# Using a fixed random_state ensures reproducibility across runs
sample_records = df_sample.sample(n=10, random_state=42)

print("--- RAW DATA SAMPLE (TRANSPOSED FOR READABILITY) ---")

# .T transposes the DataFrame (Rows become Columns, Columns become Rows)
display(sample_records.T)

--- RAW DATA SAMPLE (TRANSPOSED FOR READABILITY) ---


,6252,4684,1731,4742,4521,6340,576,5202,6363,439
ID,52018917,51988757,51796895,51990389,51984672,52020349,51448319,52001458,52020532,51323850
Codigo,2471-73-SE25,2133-71-SE25,800-11054-AG24,3663-3-AG25,1057439-175-SE25,1366597-121-TD25,4502-881-SE24,4778-8-TD25,2069-520-AG25,1549-5455-SE24
Link,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=2471-73-SE25,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=2133-71-SE25,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=800-11054-AG24,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=3663-3-AG25,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=1057439-175-SE25,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=1366597-121-TD25,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=4502-881-SE24,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=4778-8-TD25,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=2069-520-AG25,http://www.mercadopublico.cl/PurchaseOrder/Modules/PO/DetailsPurchaseOrder.aspx?codigoOC=1549-5455-SE24
Nombre,CUADERNOS Y LIBROS LICEO NARCISO TONDREAU,ORDEN DE COMPRA DESDE 2133-111-LQ22\t,JBA_CONSUMO PABELLON_800-13405-COT24,ARTICULOS DE ESCRITORIO (PEDIDO TRIMESTRAL),Determinaciones de Laboratorio Noviembre,Orden de Compra generada por Trato Directo ID 1366597-83-FTD25,PLACA RADIO DISTAL VOLAR,Orden de Compra generada por Trato Directo ID 4778-1-FTD25,SAR-CP-022025-PROGRAMA FEBRERO 2025 FARMACOS-AG,ORDEN DE COMPRA DESDE 1549-87-LQ24 SERVICIO DE SUMINISTRO OXÍGENO MEDICINAL CON ARR DE CILINDROS
Descripcion/Obervaciones,"ADQUISICIÓN DE CUADERNOS Y LIBROS PARA EL LICEO NARCISO TONDREAU, MEDIANTE EL CONTRATO DE SUMINISTRO N°2471-4-LQ24, CON CARGO A FONDO SEP.\r\r\nSOLICITUD N°17576\r\r\nÁREA: GESTIÓN DE RECURSOS\r\r\nACCIÓN: RECURSOS EDUCATIVOS\r\r\n\r\r\nOBSERVACIÓN: \r\r\n- SE DEBE ENTREGAR EN EL ESTABLECIMIENTO\r\r\n- DE ACUERDO A LA CLÁUSULA NOVENA DEL CONTRATO, ESTA INDICA QUE LAS OC EMITIDAS EN PERIODO DE VACACIONES DEBEN SER ENTREGADAS UNA VEZ ESTE OPERATIVA LA UNIDAD EDUCATIVA.\r\r\n- COORDINAR CON DIRECTOR SR MARIO MENA OJEDA FONO 42-2211152.\r\r\n-PLAZO DE ENTREGA 3 DÍAS HABILES A CONTAR DEL 03-03-2025\r\r\n-IMPUTACIÓN: 22-04-002 TEXTOS Y OTROS MATERIALES DE ENSEÑANZA.",434-22 SERV. INTERV QX TIROIDECTOMIA BILATERAL DESDE 2133-111-LQ22,CONSUMO PABELLONES QUIURGICOS\r\r\nCX: L.S.L.\r\r\nOCI =193376\r\r\n\r\r\nRazón Social: Universidad de Chile\r\r\n\r\r\nComuna: Independencia\r\r\n\r\r\nDirección: DR. CARLOS LORCA TOBAR 999\r\r\n\r\r\nGiro: ACTIVIDADES DE HOSPITALES Y CLÍNICAS PÚBLICAS\r\r\n\r\r\nRut: 60.910.000-1\r\r\n\r\r\n \r\r\n\r\r\nHorario de Atención de Lunes a Viernes entre las 08:30  13:30 // 15:00  16:30\r\r\n\r\r\nFavor enviar facturas a las siguientes casillas electrónicas dte.cl@einvoicing.signature-cloud.com / invoicing_chile@recepcion.cl en Formato XML / dte-recepcion@intercambio-planecloud.cl\r\r\n\r\r\n\r\r\n\r\r\n,"Horario de recepción de proveedores\r\r\n- Lunes a jueves 08:00 a 12:30 y 14:00 a 17:00 horas.\r\r\n- Viernes 08:00 a 13:00 horas.\r\r\n- Sábados, Domingos y festivos, no se recepcionarán pedidos en bodega.\r\r\n- Solicite orientación en OIRS y/o SOME para contactar al personal de bodega.\r\r\n- Recibe Eduardo Piña/Nakita/ Cristóbal Aro.\r\r\n- Teléfono bodega 722810390 anexo 120.\r\r\n-ADJUNTAR FACTURA ELECTRONICA, NO SE RECEPCIONARAN INSUMOS SIN FACTURA\r\r\n",Determinaciones exámenes química y hematología DESDE 1057439-41-LR21\r\r\nORDEN DE COMPRA DE INSUMOS UTILIZADOS EN MES DE noviembre EN SISTEMA DE CONCESIÓN.,GALVEZ Y ZUÑIGA SPA (LLAMADO) DEP 32,"MEMO NRO 747/10.10.2024 PABELLON Y REC DE ANESTESIA \r\r\nCÓDIGO DEL PACIENTE:EFA1137\r\r\n N° CIRUGÍA: 502062\r\r\nFECHA CIRUGÍA:07-10-

In [7]:
"""
Legal Modality Audit (Dictionary Extraction)
============================================
Purpose: Scans the entire year's datasets to extract the distinct combinations 
of Procurement Modality Codes and their official descriptions.
"""
unique_modalities = pd.DataFrame()

for file in csv_files:
    try:
        df_subset = pd.read_csv(
            file, 
            sep=';', 
            encoding='latin-1', 
            usecols=['CodigoAbreviadoTipoOC', 'DescripcionTipoOC'],
            low_memory=False
        )
        
        df_subset = df_subset.drop_duplicates()
        unique_modalities = pd.concat([unique_modalities, df_subset])
        
    except ValueError as e:
        print(f"[!] Error de columnas en {os.path.basename(file)}: {e}")
    except Exception as e:
        print(f"[!] Error leyendo {os.path.basename(file)}: {e}")

unique_modalities = unique_modalities.drop_duplicates().dropna()
unique_modalities = unique_modalities.sort_values(by='CodigoAbreviadoTipoOC').reset_index(drop=True)

print("\n--- OFFICIAL DICTIONARY OF PURCHASE METHODS (2025) ---")
display(unique_modalities)


--- OFFICIAL DICTIONARY OF PURCHASE METHODS (2025) ---


,CodigoAbreviadoTipoOC,DescripcionTipoOC
0,AG,Compra Ágil
1,CC,Compra Coordinada
2,CM,Convenio Marco
3,CT,Proveniente de compra por Cotización
4,SE,Sin emisión automática
5,TD,Proveniente de Ficha de Trato Directo


In [8]:
"""
Order Status Audit (Dictionary Extraction)
==========================================
Purpose: Scans the raw datasets to extract distinct combinations of 
Status Codes (codigoEstado) and their text descriptions (Estado).
"""
import glob
import os
import pandas as pd

raw_data_dir = "../data/raw/2025"
csv_files = glob.glob(os.path.join(raw_data_dir, "*.csv"))

unique_states = pd.DataFrame()

print("Scanning files for Status mapping...")
for file in csv_files:
    try:
        # Read only the status columns to save memory
        df_subset = pd.read_csv(
            file, 
            sep=';', 
            encoding='latin-1', 
            usecols=['codigoEstado', 'Estado'],
            low_memory=False
        )
        df_subset = df_subset.drop_duplicates()
        unique_states = pd.concat([unique_states, df_subset])
    except Exception as e:
        print(f"[!] Error reading {os.path.basename(file)}: {e}")

# Drop global duplicates and sort by status code
unique_states = unique_states.drop_duplicates().dropna()
unique_states['codigoEstado'] = pd.to_numeric(unique_states['codigoEstado'], errors='coerce')
unique_states = unique_states.sort_values(by='codigoEstado').reset_index(drop=True)

print("\n--- OFFICIAL DICTIONARY OF PURCHASE ORDER STATES (2025) ---")
display(unique_states)

Scanning files for Status mapping...

--- OFFICIAL DICTIONARY OF PURCHASE ORDER STATES (2025) ---


,codigoEstado,Estado
0,4,Enviada a proveedor
1,5,En proceso
2,6,Aceptada
3,7,Cancelacion solicitada
4,12,Recepcion Conforme


In [9]:
"""
Sanitization Preview & Impact Analysis
======================================
Purpose: Previews the impact of string regex replacement, numeric formatting, 
and primary key null-drops on a sample of the actual data.
"""

# Load a sample file (e.g., the first month)
sample_file = csv_files[0]
df_preview = pd.read_csv(sample_file, sep=';', encoding='latin-1', nrows=5000, low_memory=False)

# ---------------------------------------------------------
# 1. NUMERIC FORMATTING PREVIEW
# ---------------------------------------------------------
print("--- 1. NUMERIC FORMATTING PREVIEW ---")
# Let's create a fake column with the extreme Chilean formats you mentioned
df_preview['Test_Monto_Original'] = ['12334,349', '1.234.2134,56', '5000', '1.000.000', '0,5'] * 1000

# Fix: Remove dots (thousands), then replace commas with dots (decimals)
df_preview['Test_Monto_Cleaned'] = df_preview['Test_Monto_Original'].astype(str)\
    .str.replace('.', '', regex=False)\
    .str.replace(',', '.', regex=False)

df_preview['Test_Monto_Float'] = pd.to_numeric(df_preview['Test_Monto_Cleaned'], errors='coerce')

display(df_preview[['Test_Monto_Original', 'Test_Monto_Cleaned', 'Test_Monto_Float']].head(5))

# ---------------------------------------------------------
# 2. TEXT SANITIZATION PREVIEW
# ---------------------------------------------------------
print("\n--- 2. TEXT SANITIZATION PREVIEW ---")
text_cols = ["Nombre", "NombreProveedor"]
available_text_cols = [c for c in text_cols if c in df_preview.columns]

if available_text_cols:
    # Take a specific row to show before/after
    sample_text_before = df_preview[available_text_cols].head(3).copy()
    
    for col in available_text_cols:
        # Replace newlines, tabs, commas, and semicolons with a single space
        df_preview[col] = df_preview[col].astype(str).str.replace(r'[\n\r\t,;]+', ' ', regex=True).str.strip()
    
    sample_text_after = df_preview[available_text_cols].head(3)
    
    print("BEFORE SANITIZATION (May contain hidden \\n, \\t or commas):")
    display(sample_text_before)
    print("AFTER SANITIZATION (Clean strings for Neo4j CSV import):")
    display(sample_text_after)

# ---------------------------------------------------------
# 3. CRITICAL NULL DROPS (PRIMARY KEYS)
# ---------------------------------------------------------
print("\n--- 3. CRITICAL NULL DROPS IMPACT ---")
critical_pks = ["Codigo", "IDItem", "RutUnidadCompra", "CodigoProveedor"]
existing_pks = [col for col in critical_pks if col in df_preview.columns]

# Count how many rows are missing primary keys
initial_rows = len(df_preview)
df_valid_pks = df_preview.dropna(subset=existing_pks)
dropped_rows = initial_rows - len(df_valid_pks)

print(f"Total rows in sample: {initial_rows}")
print(f"Rows missing critical Primary Keys: {dropped_rows}")
print(f"Percentage of data lost to missing PKs: {(dropped_rows/initial_rows)*100:.2f}%")

--- 1. NUMERIC FORMATTING PREVIEW ---


,Test_Monto_Original,Test_Monto_Cleaned,Test_Monto_Float
0,"12334,349",12334.349,1.233435e+04
1,"1.234.2134,56",12342134.56,1.234213e+07
2,5000,5000,5.000000e+03
3,1.000.000,1000000,1.000000e+06
4,"0,5",0.5,5.000000e-01



--- 2. TEXT SANITIZATION PREVIEW ---
BEFORE SANITIZATION (May contain hidden \n, \t or commas):


,Nombre,NombreProveedor
0,ORDEN DE COMPRA DESDE 2560-14-LR21,CERRO DOMINADOR CSP S.A
1,ORDEN DE COMPRA DESDE 3624-28-LQ21,K D M S.A.
2,Servicio artístico musical-DIDECO-ORGANIZACIONES,VANESSA ANDREA ALVAREZ VERA


AFTER SANITIZATION (Clean strings for Neo4j CSV import):


,Nombre,NombreProveedor
0,ORDEN DE COMPRA DESDE 2560-14-LR21,CERRO DOMINADOR CSP S.A
1,ORDEN DE COMPRA DESDE 3624-28-LQ21,K D M S.A.
2,Servicio artístico musical-DIDECO-ORGANIZACIONES,VANESSA ANDREA ALVAREZ VERA



--- 3. CRITICAL NULL DROPS IMPACT ---
Total rows in sample: 5000
Rows missing critical Primary Keys: 0
Percentage of data lost to missing PKs: 0.00%
